# Recurrent dropout, stacking, and the one that does not work

Three standard refinements. Two help; the third fails, and its failure says something specific about the problem.

**Runs on:** CPU — about 25 minutes (GPU: 5 minutes) · continues from notebook 01 (same kernel) &nbsp;·&nbsp; **Slides:** [Chapter 13 — Timeseries Forecasting](../../../course-web-slides/ch13/index.html) &nbsp;·&nbsp; **Section:** 04 — Advanced use of recurrent neural networks

---

## Recurrent dropout

Ordinary dropout applied to a recurrent layer would use a **different mask at every timestep**, which destroys the signal the state is carrying. Recurrent dropout uses the *same* mask at every step, so the noise is consistent along the sequence.

In [ ]:
import keras
from keras import layers
import matplotlib.pyplot as plt

NAIVE_MAE = 2.44
sequence_length, n_features = 120, 14

inputs = keras.Input(shape=(sequence_length, n_features))
x = layers.GRU(32, recurrent_dropout=0.25)(inputs)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
cb = [keras.callbacks.ModelCheckpoint("jena_dropout.keras",
                                      save_best_only=True)]
h_do = model.fit(train_dataset, epochs=50, validation_data=val_dataset,
                 callbacks=cb, verbose=2)
mae_do = keras.models.load_model("jena_dropout.keras").evaluate(
    test_dataset, verbose=0)[1]
print(f"\nwith recurrent dropout: test MAE {mae_do:.2f}")

> ⚠️ **Fifty epochs, and it will be slow.** `recurrent_dropout` disables the cuDNN fast path, so a GPU loses most of its advantage. That is a real cost, and worth knowing before you start a run you expect to take five minutes.

## Stacking recurrent layers

In [ ]:
inputs = keras.Input(shape=(sequence_length, n_features))
x = layers.GRU(32, recurrent_dropout=0.5, return_sequences=True)(inputs)
x = layers.GRU(32, recurrent_dropout=0.5)(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)
stacked = keras.Model(inputs, outputs)

stacked.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
cb = [keras.callbacks.ModelCheckpoint("jena_stacked.keras",
                                      save_best_only=True)]
h_st = stacked.fit(train_dataset, epochs=50, validation_data=val_dataset,
                   callbacks=cb, verbose=2)
mae_st = keras.models.load_model("jena_stacked.keras").evaluate(
    test_dataset, verbose=0)[1]
print(f"\nstacked: test MAE {mae_st:.2f}")

> **Note** — **`return_sequences=True` on every layer but the last.** Without it the first GRU returns only its final state, and the second has a single vector rather than a sequence to work with — a shape error if you are lucky, a silently worse model if you are not.

## Bidirectional: the one that does not work

In [ ]:
inputs = keras.Input(shape=(sequence_length, n_features))
x = layers.Bidirectional(layers.LSTM(16))(inputs)
outputs = layers.Dense(1)(x)
bidir = keras.Model(inputs, outputs)

bidir.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
h_bi = bidir.fit(train_dataset, epochs=10, validation_data=val_dataset,
                 verbose=2)
print(f"\nbidirectional: best val MAE "
      f"{min(h_bi.history['val_mae']):.2f}   (naive {NAIVE_MAE})")

**Worse than the plain LSTM, and worse than the baseline.** This failure is informative rather than embarrassing.

A bidirectional layer runs the sequence forwards and backwards and merges the results. On text — chapter 14 — that is exactly right: the end of a sentence informs the beginning.

On weather it is not. **Chronological order is not an arbitrary convention here; it is the causal structure of the data.** The backward pass gives the model the sequence in an order that never occurs, and the extra parameters spent on it are wasted.

## Everything on one axis

In [ ]:
import numpy as np

results = [
    ("naive baseline", NAIVE_MAE),
    ("dense", 2.66),
    ("conv1d", 3.15),
    ("LSTM 16", 2.36),
    ("GRU 32 + rec. dropout", mae_do),
    ("stacked GRU", mae_st),
    ("bidirectional LSTM", min(h_bi.history["val_mae"])),
]

names = [r[0] for r in results]
vals = [r[1] for r in results]
colors = ["#888"] + ["#c0392b" if v > NAIVE_MAE else "#12b886" for v in vals[1:]]

plt.figure(figsize=(9, 4.4))
plt.barh(names[::-1], vals[::-1], color=colors[::-1])
plt.axvline(NAIVE_MAE, color="k", ls="--", lw=1.4)
plt.xlabel("test MAE (degC) -- lower is better")
plt.title("Red bars lose to a model with no parameters")
plt.tight_layout(); plt.show()

**Three of six neural networks lose to *tomorrow is like today*.** That is the most valuable result in the chapter, and it is why chapter 6 makes the baseline a required step rather than a suggestion.

## Where the remaining error lives

In [ ]:
best = keras.models.load_model("jena_stacked.keras")

preds, trues = [], []
for samples, targets in test_dataset.take(20):
    preds.append(best.predict(samples, verbose=0).ravel())
    trues.append(np.array(targets))
preds = np.concatenate(preds); trues = np.concatenate(trues)
err = np.abs(preds - trues)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.2))
a1.scatter(trues, preds, s=5, alpha=.35)
lims = [trues.min() - 2, trues.max() + 2]
a1.plot(lims, lims, "k--", lw=1)
a1.set_xlabel("actual (degC)"); a1.set_ylabel("predicted")
a1.set_title("Predicted against actual"); a1.set_aspect("equal")

a2.scatter(trues, err, s=5, alpha=.35)
a2.set_xlabel("actual temperature (degC)"); a2.set_ylabel("absolute error")
a2.set_title("Error is largest at the extremes")
plt.tight_layout(); plt.show()

print(f"MAE overall:      {err.mean():.2f}")
cold = trues < np.percentile(trues, 10)
hot = trues > np.percentile(trues, 90)
print(f"MAE coldest 10%:  {err[cold].mean():.2f}")
print(f"MAE hottest 10%:  {err[hot].mean():.2f}")

The error concentrates at the extremes — which are exactly the cases a weather forecast is most needed for. **A single MAE hides that entirely**, and it is the same lesson as chapter 11's IoU per class: report the metric where it matters, not only on average.

---

## What to take away

- Recurrent dropout needs a consistent mask across timesteps, and it disables the fast cuDNN path.
- Stack with `return_sequences=True` on every layer but the last.
- **Bidirectional fails here** — chronological order is the causal structure, not a convention.
- Half the neural networks in this chapter lose to a parameterless baseline. Compute the baseline.